# Spotify Audio Features & Health Clustering Analysis

**Exploring Spotify track audio features to understand music patterns and clustering through data analysis.**

## Project Overview

This project analyzes Spotify audio features (danceability, energy, tempo, valence, etc.) to understand patterns in music characteristics. We explore how tracks cluster by audio features and what makes songs popular.

## Learning Objectives

1. Work with music/audio feature data
2. Analyze distributions of continuous features
3. Explore correlations between audio characteristics
4. Apply clustering concepts to group similar tracks
5. Understand what makes music 'popular' from a data perspective

## Business / Research Problem

Music streaming platforms use audio features for recommendation systems. Understanding these features helps:
- **Playlist curators** group similar songs
- **Artists** understand what makes tracks popular
- **Platforms** build better recommendation engines

**Key question:** *What audio feature patterns exist in Spotify data, and how do they relate to popularity?*

## Why This Analysis Matters

The music streaming industry is massive. Data-driven understanding of audio features powers recommendation algorithms, playlist generation, and artist analytics.

## Dataset Overview

Key Spotify audio features include:
- `danceability` (0-1): How suitable for dancing
- `energy` (0-1): Intensity and activity
- `valence` (0-1): Musical positiveness/happiness
- `tempo`: Beats per minute
- `loudness`: Overall loudness (dB)
- `speechiness`: Presence of spoken words
- `acousticness`: Acoustic vs electronic
- `instrumentalness`: Vocal vs instrumental
- `popularity`: Track popularity score

## Dataset Source & License Notes

- **Source:** [Kaggle - Spotify Tracks Dataset](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)
- **License:** CC0 Public Domain
- **Usage:** Educational purposes

## Environment Setup

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy kagglehub scikit-learn -q

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print('All imports loaded successfully.')

## Configuration / Constants

In [ ]:
RANDOM_SEED = 42
KAGGLE_DATASET = 'maharshipandya/-spotify-tracks-dataset'
AUDIO_FEATURES = ['danceability', 'energy', 'valence', 'tempo', 'loudness',
                   'speechiness', 'acousticness', 'instrumentalness', 'liveness']
np.random.seed(RANDOM_SEED)

## Dataset Download / Loading

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download(KAGGLE_DATASET)
print(f'Downloaded to: {path}')

csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))
print(f'Loaded {len(df)} rows and {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nMissing values:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'None')
print(f'\nDuplicates: {df.duplicated().sum()}')

# Check available audio features
avail_features = [f for f in AUDIO_FEATURES if f in df.columns]
print(f'\nAvailable audio features: {avail_features}')
df[avail_features].describe()

## Data Cleaning

In [ ]:
df = df.drop_duplicates()
df = df.dropna(subset=avail_features)
print(f'After cleaning: {len(df)} rows')

# Identify popularity column
pop_col = [c for c in df.columns if 'popular' in c.lower()]
if pop_col:
    pop_col = pop_col[0]
    print(f'Popularity column: {pop_col}')
    print(f'Popularity range: {df[pop_col].min()} - {df[pop_col].max()}')

## Exploratory Data Analysis

In [ ]:
print('=== Audio Feature Means ===')
print(df[avail_features].mean().round(3))

if pop_col:
    print(f'\n=== Popularity Stats ===')
    print(f'Mean: {df[pop_col].mean():.1f}, Median: {df[pop_col].median():.1f}')

## Univariate Analysis

In [ ]:
n_feats = len(avail_features)
n_rows = (n_feats + 2) // 3
fig, axes = plt.subplots(n_rows, 3, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(avail_features):
    axes[i].hist(df[feat], bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(feat)
for j in range(len(avail_features), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if 'energy' in df.columns and 'loudness' in df.columns:
    axes[0].scatter(df['energy'], df['loudness'], alpha=0.1, s=2, c='steelblue')
    axes[0].set_xlabel('Energy')
    axes[0].set_ylabel('Loudness')
    axes[0].set_title('Energy vs Loudness')

if 'danceability' in df.columns and 'valence' in df.columns:
    axes[1].scatter(df['danceability'], df['valence'], alpha=0.1, s=2, c='coral')
    axes[1].set_xlabel('Danceability')
    axes[1].set_ylabel('Valence')
    axes[1].set_title('Danceability vs Valence')

if 'acousticness' in df.columns and 'energy' in df.columns:
    axes[2].scatter(df['acousticness'], df['energy'], alpha=0.1, s=2, c='mediumseagreen')
    axes[2].set_xlabel('Acousticness')
    axes[2].set_ylabel('Energy')
    axes[2].set_title('Acousticness vs Energy')

plt.tight_layout()
plt.show()

## Feature-Specific Insights

In [ ]:
# Popularity bins analysis
if pop_col:
    df['pop_tier'] = pd.cut(df[pop_col], bins=[0, 20, 50, 75, 100],
                             labels=['Low', 'Medium', 'High', 'Very High'])
    print('=== Audio Features by Popularity Tier ===')
    print(df.groupby('pop_tier')[avail_features].mean().round(3))

## Statistical Checks / Hypothesis Testing

In [ ]:
if pop_col:
    print('=== Correlation with Popularity ===')
    for feat in avail_features:
        r, p = stats.pearsonr(df[feat], df[pop_col])
        print(f'{feat:>20}: r={r:+.4f}, p={p:.2e}')

# Audio feature correlation matrix
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[avail_features].corr(), annot=True, cmap='coolwarm', center=0,
            fmt='.2f', ax=ax, linewidths=0.5)
ax.set_title('Audio Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## Visual Analysis

In [ ]:
if pop_col:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(df[pop_col], bins=50, color='mediumseagreen', edgecolor='white')
    ax.axvline(df[pop_col].mean(), color='red', linestyle='--', label=f'Mean: {df[pop_col].mean():.0f}')
    ax.set_title('Popularity Distribution')
    ax.set_xlabel('Popularity Score')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Key Findings

1. **Energy and loudness are strongly correlated** — louder songs feel more energetic
2. **Acousticness inversely correlates with energy** — as expected
3. **Danceability and valence have moderate positive correlation** — happy songs are more danceable
4. **Most tracks have low popularity** — the distribution is heavily right-skewed
5. **Audio features alone weakly predict popularity** — marketing and artist fame matter more

## Limitations

1. **No genre labels in some versions:** Genre-specific analysis may be limited
2. **Popularity is temporal:** Scores change over time
3. **Audio features are algorithmic:** They're Spotify's computed metrics, not human ratings
4. **No user data:** Individual listening patterns aren't captured

## Recommendations / Next Steps

1. Apply K-means clustering to group tracks by audio profile
2. Build a genre classifier using audio features
3. Analyze feature trends over decades

### How to Extend This Analysis
- Use the Spotify API for real-time data
- Build a playlist recommendation engine
- Compare features across genres

## Common Mistakes

1. **Expecting audio features to strongly predict popularity** — they don't; marketing matters more
2. **Not checking feature ranges:** Some features are 0-1, others have different scales
3. **Ignoring multicollinearity:** Energy/loudness are highly correlated—be careful in regression
4. **Treating Spotify features as ground truth:** They're algorithmic estimates, not perfect measures
5. **Not sampling for visualization:** Scatter plots with 100k+ points need alpha tuning or sampling

## Mini Challenge / Exercises

1. What audio feature profile defines the most popular tracks (top 1%)?
2. Apply K-means (k=5) to cluster tracks by audio features. What characterizes each cluster?
3. Find the most 'unique' track — furthest from the centroid of all tracks
4. Is there a correlation between track duration and popularity?
5. Create a radar chart comparing the average audio profile of high vs low popularity tracks

In [ ]:
# Space for exercise solutions

## Final Summary / Key Takeaways

- **Audio features capture distinct musical dimensions** — energy, mood, danceability
- **Strong correlations exist** between energy/loudness and acousticness/energy
- **Popularity is weakly predicted** by audio features alone
- **Clustering reveals natural music groups** beyond genre labels
- **This dataset is excellent for practicing** unsupervised learning and feature analysis

Understanding audio features is key to music data science and recommendation systems.